# Deploy and score a machine learning model by using an online endpoint 

Learn how to use an online endpoint to deploy your model, so you don't have to create and manage the underlying infrastructure. You'll begin by deploying a model on your local machine to debug any errors, and then you'll deploy and test it in Azure.

Managed online endpoints help to deploy your ML models in a turnkey manner. Managed online endpoints work with powerful CPU and GPU machines in Azure in a scalable, fully managed way. Managed online endpoints take care of serving, scaling, securing, and monitoring your models, freeing you from the overhead of setting up and managing the underlying infrastructure. 

For more information, see [What are Azure Machine Learning endpoints?](https://learn.microsoft.com/azure/machine-learning/concept-endpoints), and [Deploy an ML model with an online endpoint](https://learn.microsoft.com/azure/machine-learning/how-to-deploy-online-endpoints).

## Prerequisites

* To use Azure Machine Learning, you must have an Azure subscription. If you don't have an Azure subscription, create a free account before you begin. Try the [free or paid version of Azure Machine Learning](https://azure.microsoft.com/free/).

* Install and configure the [Python SDK v2](sdk/setup.sh).

* You must have an Azure resource group, and you (or the service principal you use) must have Contributor access to it.

* You must have an Azure Machine Learning workspace. 

* To deploy locally, you must install Docker Engine on your local computer. We highly recommend this option, so it's easier to debug issues.

In [1]:
#%pip install docker

# 1. Connect to Azure Machine Learning Workspace

The [workspace](https://docs.microsoft.com/en-us/azure/machine-learning/concept-workspace) is the top-level resource for Azure Machine Learning, providing a centralized place to work with all the artifacts you create when you use Azure Machine Learning. In this section we will connect to the workspace in which the job will be run.

## 1.1. Import the required libraries

In [ ]:
# import required libraries
from azure.ai.ml import MLClient
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    Model,
    Environment,
    CodeConfiguration,
)
from azure.ai.ml.constants import AssetTypes
from azure.identity import AzureCliCredential

## 1.2. Configure workspace details and get a handle to the workspace

To connect to a workspace, we need identifier parameters - a subscription, resource group and workspace name. We will use these details in the `MLClient` from `azure.ai.ml` to get a handle to the required Azure Machine Learning workspace. We use the default [default azure authentication](https://docs.microsoft.com/en-us/python/api/azure-identity/azure.identity.defaultazurecredential?view=azure-python) for this tutorial. Check the [configuration notebook](../../jobs/configuration.ipynb) for more details on how to configure credentials and connect to a workspace.

In [2]:
# enter details of your AML workspace
subscription_id = "710c48d7-7060-4d97-9be0-699f76c25447"
resource_group = "rg-gst-dev-ussc-01"
workspace = "ml-gst-dev-usscc-01"

In [ ]:
# get a handle to the workspace
ml_client = MLClient(
    AzureCliCredential(), subscription_id, resource_group, workspace
)

## Deploy and debug locally by using local endpoints

### Note
* To deploy locally, [Docker Engine](https://docs.docker.com/engine/install/) must be installed.
* Docker Engine must be running. Docker Engine typically starts when the computer starts. If it doesn't, you can [troubleshoot Docker Engine](https://docs.docker.com/config/daemon/#start-the-daemon-manually).

# 2. Define endpoint and deployment

## 2.1 Define the endpoint

To define an endpoint, you need to specify:

* Endpoint name: The name of the endpoint. It must be unique in the Azure region. For more information on the naming rules, see [managed online endpoint limits](how-to-manage-quotas.md#azure-machine-learning-managed-online-endpoints).
* Authentication mode: The authentication method for the endpoint. Choose between key-based authentication and Azure Machine Learning token-based authentication. A key doesn't expire, but a token does expire. For more information on authenticating, see [Authenticate to an online endpoint](how-to-authenticate-online-endpoint.md).
* Optionally, you can add a description and tags to your endpoint.

In [4]:
# Define an endpoint name
endpoint_name =  "gst-ner-endpoint-dev"

# Example way to define a random name
import datetime


# create an online endpoint
endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="GST GPT NER Model endpoint",
    auth_mode="key",
    public_network_access="disabled",
    tags={"Model": "GPTv12"},
)

## 2.2 Define the deployment

A deployment is a set of resources required for hosting the model that does the actual inferencing. To deploy a model, you must have:

- Model files (or the name and version of a model that's already registered in your workspace). In the example, we have a scikit-learn model that does regression.
- A scoring script, that is, code that executes the model on a given input request. The scoring script receives data submitted to a deployed web service and passes it to the model. The script then executes the model and returns its response to the client. The scoring script is specific to your model and must understand the data that the model expects as input and returns as output. In this example, we have a *score.py* file.
- An environment in which your model runs. The environment can be a Docker image with Conda dependencies or a Dockerfile.
- Settings to specify the instance type and scaling capacity.

The following table describes the key attributes of a deployment:

| Attribute      | Description                                                                                                                                                                                                                                                                                                                                                                                    |
|-----------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| Name           | The name of the deployment.                                                                                                                                                                                                                                                                                                                                                                    |
| Endpoint name  | The name of the endpoint to create the deployment under.                                                                                                                                                                                                                                                                                                                                       |
| Model          | The model to use for the deployment. This value can be either a reference to an existing versioned model in the workspace or an inline model specification.                                                                                                                                                                                                                                    |
| Code path      | The path to the directory on the local development environment that contains all the Python source code for scoring the model. You can use nested directories and packages.                                                                                                                                                                                                                    |
| Scoring script | The relative path to the scoring file in the source code directory. This Python code must have an `init()` function and a `run()` function. The `init()` function will be called after the model is created or updated (you can use it to cache the model in memory, for example). The `run()` function is called at every invocation of the endpoint to do the actual scoring and prediction. |
| Environment    | The environment to host the model and code. This value can be either a reference to an existing versioned environment in the workspace or an inline environment specification.                                                                                                                                                                                                                 |
| Instance type  | The VM size to use for the deployment. For the list of supported sizes, see [Managed online endpoints SKU list](reference-managed-online-endpoints-vm-sku-list.md).                                                                                                                                                                                                                            |
| Instance count | The number of instances to use for the deployment. Base the value on the workload you expect. For high availability, we recommend that you set the value to at least `3`. We reserve an extra 20% for performing upgrades. For more information, see [managed online endpoint quotas](how-to-manage-quotas.md#azure-machine-learning-managed-online-endpoints).                                |

In [6]:
# #Register the model on Azure
# from azureml.core import Workspace, Model

# # Assuming you already have your Workspace set up
# ws = Workspace.from_config()

# model = Model.register(workspace=ws,
#                        model_name="gst-ner-model",
#                        model_path="./dependencies", 
#                        description="GST NER model with dependencies")

# # file_model = Model(
# #     path="./ner-model-2/model/GST NER Model",
# #     type=AssetTypes.CUSTOM_MODEL,
# #     name="gst-ner-model",
# #     description="GST NER model with dependencies.",
# # )
# # ml_client.models.create_or_update(file_model)


Registering model gst-ner-model


In [5]:
models = ml_client.models.list()
for model in models:
    print(model.name)

GST-NER-Model
gst-ner-model-test
gst-ner-model


Retrying due to transient client side error HTTPSConnectionPool(host='dc.services.visualstudio.com', port=443): Max retries exceeded with url: /v2.1/track (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7f3c20512610>: Failed to establish a new connection: [Errno -2] Name or service not known')).


In [5]:
model = Model(path="./dependencies")

env = Environment(
    conda_file="./environment/conda.yaml",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
)

blue_deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=endpoint_name,
    model=model,
    environment=env,
    code_configuration=CodeConfiguration(
        code="./onlinescoring", scoring_script="score.py"
    ),
    instance_type="Standard_DS3_v2",
    instance_count=1,
    egress_public_network_access="disabled"
)

# 3. Create local endpoint and deployment

## 3.1 Create local endpoint

The goal of a local endpoint deployment is to validate and debug your code and configuration before you deploy to Azure. Local deployment has the following limitations:
* Local endpoints *do not support* traffic rules, authentication, or probe settings.
* Local endpoints support only one deployment per endpoint.
* They support local model files only. If you want to test registered models, first download them, then use `path` in the deployment definition to refer to the parent folder.

In [6]:
ml_client.online_endpoints.begin_create_or_update(endpoint, local=True)

Updating local endpoint (gst-ner-endpoint-dev) Done (0m 0s)


ManagedOnlineEndpoint({'public_network_access': 'disabled', 'provisioning_state': 'Failed', 'scoring_uri': None, 'openapi_uri': None, 'name': 'gst-ner-endpoint-dev', 'description': 'GST GPT NER Model endpoint', 'tags': {'Model': 'GPTv12'}, 'properties': {}, 'print_as_yaml': True, 'id': None, 'Resource__source_path': None, 'base_path': PosixPath('/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-rahul-cpu01/code/Users/ADMDSCRS3/NLP-NER-Model-API'), 'creation_context': None, 'serialize': <msrest.serialization.Serializer object at 0x7fcf0430b3d0>, 'auth_mode': 'key', 'location': 'local', 'identity': None, 'traffic': {}, 'mirror_traffic': {}, 'kind': None})

Retrying due to transient client side error HTTPSConnectionPool(host='dc.services.visualstudio.com', port=443): Max retries exceeded with url: /v2.1/track (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fcf0430b640>: Failed to establish a new connection: [Errno -2] Name or service not known')).


## 3.2 Create local deployment

Now, create a deployment named `blue` under the endpoint.

In [7]:
ml_client.online_deployments.begin_create_or_update(
    deployment=blue_deployment, local=True
)

Updating local deployment (gst-ner-endpoint-dev / blue) .
Building Docker image from Dockerfile
Step 1/6 : FROM mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest
 ---> 7b2780449df8
Step 2/6 : RUN mkdir -p /var/azureml-app/
 ---> Using cache
 ---> 7feb5db6abf2
Step 3/6 : WORKDIR /var/azureml-app/
 ---> Using cache
 ---> 66bf02490521
Step 4/6 : COPY conda.yml /var/azureml-app/
 ---> 24f795201ca5
Step 5/6 : RUN conda env create -n inf-conda-env --file conda.yml
 ---> Running in 6de19515ced2
.Retrieving notices: ...working... done
Channels:
 - conda-forge
 - defaults
Platform: linux-64
...done
Solving environment: ...working... .done

Preparing transaction: ...working... done
Verifying transaction: ...working... .done
Executing transaction: ...working... done
Installing pip dependencies: ...working... .............Ran pip subprocess with arguments:
['/opt/miniconda/envs/inf-conda-env/bin/python', '-m', 'pip', 'install', '-U', '-r', '/var/azureml-app/condaenv.bk939xkq.requirements.t

ManagedOnlineDeployment({'private_network_connection': None, 'provisioning_state': 'Succeeded', 'endpoint_name': 'gst-ner-endpoint-dev', 'type': 'Managed', 'name': 'blue', 'description': None, 'tags': {}, 'properties': {}, 'print_as_yaml': True, 'id': None, 'Resource__source_path': None, 'base_path': PosixPath('/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-rahul-cpu01/code/Users/ADMDSCRS3/NLP-NER-Model-API'), 'creation_context': None, 'serialize': <msrest.serialization.Serializer object at 0x7fcf340c6850>, 'model': Model({'job_name': None, 'intellectual_property': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': '015d227d9052c78b4f91890db26610a3', 'description': None, 'tags': {}, 'properties': {}, 'print_as_yaml': True, 'id': None, 'Resource__source_path': None, 'base_path': PosixPath('/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-rahul-cpu01/code/Users/ADMDSCRS3/NLP-NER-Model-API'), 'creation_context': None, 'serialize': <msr

The `local=True` flag directs the SDK to deploy the endpoint in the Docker environment.

# 4. Verify the local deployment succeeded

## 4.1 Check the status to see whether the model was deployed without error

In [8]:
ml_client.online_endpoints.get(name=endpoint_name, local=True)

ManagedOnlineEndpoint({'public_network_access': 'disabled', 'provisioning_state': 'Succeeded', 'scoring_uri': 'http://localhost:32770/score', 'openapi_uri': None, 'name': 'gst-ner-endpoint-dev', 'description': 'GST GPT NER Model endpoint', 'tags': {'Model': 'GPTv12'}, 'properties': {}, 'print_as_yaml': True, 'id': None, 'Resource__source_path': None, 'base_path': PosixPath('/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-rahul-cpu01/code/Users/ADMDSCRS3/NLP-NER-Model-API'), 'creation_context': None, 'serialize': <msrest.serialization.Serializer object at 0x7fcf04283c70>, 'auth_mode': 'key', 'location': 'local', 'identity': None, 'traffic': {}, 'mirror_traffic': {}, 'kind': None})

## 4.2 Get logs

In [9]:
print(ml_client.online_deployments.get_logs(
    name="blue", endpoint_name=endpoint_name, local=True, lines=1000
))

2024-04-24T10:57:21,263881567+00:00 - rsyslog/run 
2024-04-24T10:57:21,263937067+00:00 - gunicorn/run 
2024-04-24T10:57:21,264992972+00:00 | gunicorn/run | 
2024-04-24T10:57:21,266161177+00:00 | gunicorn/run | ###############################################
2024-04-24T10:57:21,267230782+00:00 | gunicorn/run | AzureML Container Runtime Information
2024-04-24T10:57:21,268243186+00:00 | gunicorn/run | ###############################################
2024-04-24T10:57:21,269311691+00:00 | gunicorn/run | 
2024-04-24T10:57:21,278337931+00:00 - nginx/run 
2024-04-24T10:57:21,552402018+00:00 | gunicorn/run | 
2024-04-24T10:57:21,560052550+00:00 | gunicorn/run | AzureML image information: openmpi4.1.0-ubuntu20.04, Materializaton Build:20240418.v1
2024-04-24T10:57:21,561299455+00:00 | gunicorn/run | 
2024-04-24T10:57:21,562598660+00:00 | gunicorn/run | 
2024-04-24T10:57:21,563832165+00:00 | gunicorn/run | PATH environment variable: /opt/miniconda/envs/inf-conda-env/bin:/opt/miniconda/condabin:/opt

## 4.3 Invoke the local endpoint
Invoke the endpoint to score the model by using the convenience command invoke and passing query parameters that are stored in a JSON file

In [10]:
%%time
ml_client.online_endpoints.invoke(
    endpoint_name=endpoint_name,
    request_file="./sample-request.json",
    local=True,
)

CPU times: user 5.06 ms, sys: 2.86 ms, total: 7.91 ms
Wall time: 1.57 s


'{"userInput": {"searchQuery": "gf30, high density"}, "modelOutput": {"entities": {"GRADE": [], "APPLICATION": [], "BRAND": [], "POLYMER": [], "PROPERTY": [{"property_name": "density", "modifier": {"value": "high", "min": null, "max": null, "unit": ""}, "property_type": "property"}], "FILLER": [{"filler_name": ["glass fiber"]}, {"total_load": {"value": "30", "min": "25", "max": "35"}}], "FEATURE": [], "PROCESSING": [], "DELIVERY_FORM": [], "COMPETITOR_GRADE": [], "AUTO_CERT": [], "RAILWAY_CERT": [], "WATER_CERT": [], "NSF_CERT": []}, "unidentified": ""}, "modelVersion": "GPTv12_04_19_24", "apiVersion": "v2.0.0"}'

# 5. Deploy your online endpoint to Azure
Next, deploy your online endpoint to Azure.

## 5.1 Create the endpoint
Using the `endpoint` we defined earlier and the `MLClient` created earlier, we'll now create the endpoint in the workspace. This command will start the endpoint creation and return a confirmation response while the endpoint creation continues.

In [11]:
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

Retrying due to transient client side error HTTPSConnectionPool(host='dc.services.visualstudio.com', port=443): Max retries exceeded with url: /v2.1/track (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fcf0422f130>: Failed to establish a new connection: [Errno -2] Name or service not known')).
Retrying due to transient client side error HTTPSConnectionPool(host='dc.services.visualstudio.com', port=443): Max retries exceeded with url: /v2.1/track (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fcf04036730>: Failed to establish a new connection: [Errno -2] Name or service not known')).


ManagedOnlineEndpoint({'public_network_access': 'Disabled', 'provisioning_state': 'Succeeded', 'scoring_uri': 'https://gst-ner-endpoint-dev.southcentralus.inference.ml.azure.com/score', 'openapi_uri': 'https://gst-ner-endpoint-dev.southcentralus.inference.ml.azure.com/swagger.json', 'name': 'gst-ner-endpoint-dev', 'description': 'GST GPT NER Model endpoint', 'tags': {'Model': 'GPTv12', 'Application': 'GST', 'CostCenter': '1125500', 'DataSensitivity': '', 'Department': '', 'DRTier': '', 'Environment': '', 'Function': '', 'ManagedBy': '', 'OwnedBy': '', 'ProjectID': ''}, 'properties': {'azureml.onlineendpointid': '/subscriptions/710c48d7-7060-4d97-9be0-699f76c25447/resourcegroups/rg-gst-dev-ussc-01/providers/microsoft.machinelearningservices/workspaces/ml-gst-dev-usscc-01/onlineendpoints/gst-ner-endpoint-dev', 'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/710c48d7-7060-4d97-9be0-699f76c25447/providers/Microsoft.MachineLearningServices/locations/southcentralus/mfeO

In [13]:
ml_client.online_deployments.begin_create_or_update(blue_deployment).result()

Check: endpoint gst-ner-endpoint-dev exists
Your file exceeds 100 MB. If you experience low speeds, latency, or broken connections, we recommend using the AzCopyv10 tool for this file transfer.

Example: azcopy copy '/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-rahul-cpu01/code/Users/ADMDSCRS3/NLP-NER-Model-API/dependencies' 'https://sagmlstoredev01.blob.core.windows.net/azureml-blobstore-e3d05e55-9be9-4bc9-a48a-257b67f56289/LocalUpload/607aeb9589220a4473d78cffc10422bf/dependencies' 

See https://docs.microsoft.com/azure/storage/common/storage-use-azcopy-v10 for more information.


Exception: You don't have permission to alter this storage account. Ensure that you have been assigned both Storage Blob Data Reader and Storage Blob Data Contributor roles.

In [ ]:
# blue deployment takes 100 traffic
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

In [6]:
# azure.identity.__version__ : '1.13.0'

In [9]:
import azure.core
# azure.core.__version__ '1.26.4'

'1.26.4'

In [2]:
!pip install azure-identity==1.13.0

     |████████████████████████████████| 151 kB 3.3 MB/s eta 0:00:01
ERROR: azureml-inference-server-http 0.8.4 has requirement flask<2.3.0, but you'll have flask 2.3.2 which is incompatible.
  Attempting uninstall: azure-identity
    Found existing installation: azure-identity 1.16.0
    Uninstalling azure-identity-1.16.0:
      Successfully uninstalled azure-identity-1.16.0


In [1]:
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default"
)

client = AzureOpenAI(
    api_version='2023-07-01-preview',
    azure_endpoint='https://oai-gst-d-usnc-01.openai.azure.com/',
    azure_ad_token_provider=token_provider
)


In [2]:
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default"
)

client = AzureOpenAI(
    api_version="2024-02-15-preview",
    azure_endpoint='https://oai-gst-d-usnc-01.openai.azure.com/',
    azure_ad_token_provider=token_provider
)

response = client.chat.completions.create(
    model="gpt-35-turbo-0125", # model = "deployment_name".
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Does Azure OpenAI support customer managed keys?"},
        {"role": "assistant", "content": "Yes, customer managed keys are supported by Azure OpenAI."},
        {"role": "user", "content": "Do other Azure AI services support this too?"}
    ]
)

print(response.choices[0].message.content)

DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	ManagedIdentityCredential: request() got an unexpected keyword argument 'enable_cae'
To mitigate this issue, please refer to the troubleshooting guidelines here at https://aka.ms/azsdk/python/identity/defaultazurecredential/troubleshoot.


ClientAuthenticationError: DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	ManagedIdentityCredential: request() got an unexpected keyword argument 'enable_cae'
To mitigate this issue, please refer to the troubleshooting guidelines here at https://aka.ms/azsdk/python/identity/defaultazurecredential/troubleshoot.

In [5]:
content = "Act as an NER model trained on the data corpus of 'Celanese' which is a global chemical leader in the production of differentiated chemistry solutions and specialty materials used in most major industries and consumer applications. Ensure the output is a structured dictionary format."
fine_tuned_model_id = "gpt-35-turbo-0613: ftjob-1b394e8df8ac44e4af410c3d82604bc6-NERv12"
engine = "oai-gpt35-NERv12-gst-d-usnc-01"

query = "gf30"

response = client.chat.completions.create(
    seed=12,
    temperature = 0.2,
    model=fine_tuned_model_id,
    messages=[
        {"role": "system", "content": content}, 
        {"role": "user", "content": query},
        ],
)

print(response.choices[0].message.content)

DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	ManagedIdentityCredential: request() got an unexpected keyword argument 'enable_cae'
To mitigate this issue, please refer to the troubleshooting guidelines here at https://aka.ms/azsdk/python/identity/defaultazurecredential/troubleshoot.


ClientAuthenticationError: DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	ManagedIdentityCredential: request() got an unexpected keyword argument 'enable_cae'
To mitigate this issue, please refer to the troubleshooting guidelines here at https://aka.ms/azsdk/python/identity/defaultazurecredential/troubleshoot.